## Create and Optimize Unity Catalog Tables for your Genie Agents

### Create the Schema

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS YOUR_UNITY_CATALOG_NAME.genie_lab;

### Set the Execution Context

In [ ]:
%sql
USE SCHEMA genie_lab;

### Create the Customers Table

In [ ]:
%sql
CREATE OR REPLACE TABLE customers (
    customer_id STRING NOT NULL,
    customer_name STRING,
    customer_segment STRING,
    region STRING,
    signup_date DATE,

    CONSTRAINT customers_pk
        PRIMARY KEY (customer_id) NOT ENFORCED
);

INSERT INTO customers VALUES
('C001', 'Acme Corp',       'Enterprise', 'West',  '2025-01-10'),
('C002', 'Nova Systems',    'SMB',        'East',  '2025-03-15'),
('C003', 'Orion Labs',      'Enterprise', 'West',  '2025-04-20'),
('C004', 'BluePeak Ltd',    'Mid-Market', 'South', '2025-05-12'),
('C005', 'Vertex AI',       'Enterprise', 'East',  '2025-06-18'),
('C006', 'Summit Retail',   'SMB',        'West',  '2025-07-22'),
('C007', 'Nimbus Corp',     'Mid-Market', 'North', '2025-08-05'),
('C008', 'Quantum Works',   'Enterprise', 'South', '2025-09-14');

### Create the Products Table

In [ ]:
%sql
CREATE OR REPLACE TABLE products (
    product_id STRING NOT NULL,
    product_name STRING,
    product_category STRING,
    list_price DOUBLE,

    CONSTRAINT products_pk
        PRIMARY KEY (product_id) NOT ENFORCED
);

INSERT INTO products VALUES
('P101', 'AI Starter',       'AI',    1200),
('P102', 'AI Enterprise',    'AI',    5000),
('P103', 'Cloud Basic',      'Cloud',  800),
('P104', 'Cloud Enterprise', 'Cloud', 4000),
('P105', 'Data Starter',     'Data',  1500),
('P106', 'Data Enterprise',  'Data',  6000);

### Create the Orders Table

In [ ]:
%sql
CREATE OR REPLACE TABLE orders (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    quantity INT,
    gross_revenue DOUBLE,
    discount_amount DOUBLE,
    order_status STRING,
    returned BOOLEAN
);

INSERT INTO orders VALUES
('O001','C001','P102','2026-01-10',2,10000,500,'COMPLETED',false),
('O002','C002','P103','2026-01-15',4,3200,200,'COMPLETED',true),
('O003','C003','P106','2026-02-03',1,6000,0,'COMPLETED',false),
('O004','C004','P105','2026-02-14',3,4500,500,'COMPLETED',false),
('O005','C005','P102','2026-03-01',2,10000,1000,'COMPLETED',false),
('O006','C006','P101','2026-03-11',2,2400,100,'CANCELLED',false),

('O007','C001','P104','2026-04-05',2,8000,500,'COMPLETED',false),
('O008','C007','P105','2026-04-19',4,6000,300,'COMPLETED',true),
('O009','C008','P106','2026-05-08',2,12000,500,'COMPLETED',false),
('O010','C003','P102','2026-05-20',1,5000,250,'COMPLETED',false),
('O011','C002','P103','2026-06-02',3,2400,100,'COMPLETED',true),
('O012','C005','P106','2026-06-21',1,6000,0,'COMPLETED',false),

('O013','C001','P102','2026-07-04',3,15000,1000,'COMPLETED',false),
('O014','C004','P105','2026-07-15',2,3000,200,'COMPLETED',true),
('O015','C006','P101','2026-08-01',5,6000,500,'COMPLETED',false),
('O016','C008','P104','2026-08-17',2,8000,0,'COMPLETED',false),

('O017','C003','P106','2026-09-03',2,12000,750,'COMPLETED',false),
('O018','C005','P102','2026-09-12',2,10000,500,'COMPLETED',false);

### Add Table Context

In [ ]:
%sql
COMMENT ON TABLE customers IS
'Customer master data. Each row represents one customer and contains their business segment and sales region.';

COMMENT ON TABLE products IS
'Product catalog containing product names, product categories, and standard list prices.';

COMMENT ON TABLE orders IS
'Sales transaction table. Each row represents an order. Use completed orders for sales and revenue analysis. Cancelled orders must not contribute to revenue.';

### Add Column-level Semantics

In [ ]:
%sql
ALTER TABLE orders
ALTER COLUMN gross_revenue
COMMENT 'Revenue generated before applying discounts.';

ALTER TABLE orders
ALTER COLUMN discount_amount
COMMENT 'Discount applied to the order. Net revenue equals gross revenue minus discount amount.';

ALTER TABLE orders
ALTER COLUMN order_status
COMMENT 'Status of the order. Only COMPLETED orders should contribute to sales metrics.';

ALTER TABLE orders
ALTER COLUMN returned
COMMENT 'True when a completed order was returned by the customer.';

ALTER TABLE customers
ALTER COLUMN customer_segment
COMMENT 'Business classification of the customer: SMB, Mid-Market, or Enterprise.';

ALTER TABLE customers
ALTER COLUMN region
COMMENT 'Sales territory assigned to the customer.';

### Establish Table Relationships

In [ ]:
%sql
ALTER TABLE customers
ADD CONSTRAINT customers_pk
PRIMARY KEY(customer_id) NOT ENFORCED;

ALTER TABLE products
ADD CONSTRAINT products_pk
PRIMARY KEY(product_id) NOT ENFORCED;

In [ ]:
%sql
ALTER TABLE orders
ADD CONSTRAINT orders_customer_fk
FOREIGN KEY(customer_id)
REFERENCES customers(customer_id)
NOT ENFORCED;

ALTER TABLE orders
ADD CONSTRAINT orders_product_fk
FOREIGN KEY(product_id)
REFERENCES products(product_id)
NOT ENFORCED;